# 🎗️ Breast Cancer Detection — Machine Learning Lab
### Predict Malignant vs Benign Tumors with Supervised Learning

---

**What You Will Do Today:**
- 📦 Load the Wisconsin Breast Cancer dataset
- 🔍 Explore the medical features (EDA)
- 🧹 Preprocess & scale the data
- 🤖 Train **6 different classification models**
- 📊 Compare them — and learn why **Recall matters most in medicine**

---

> **Target:** Predict whether a tumor is **Malignant (0)** or **Benign (1)**

---

## 📌 SECTION 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset
from sklearn.datasets import load_breast_cancer

# Preprocessing & splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# The 6 classification models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc,
                             precision_score, recall_score, f1_score)

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## 📌 SECTION 2: Load the Dataset

The Wisconsin Breast Cancer dataset is built into sklearn — no download needed.
Features are measurements taken from digitized images of breast mass cell nuclei.

In [ ]:
# Load the dataset
data = load_breast_cancer()

# Build a DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print('Dataset shape:', df.shape)
print('\nTarget meaning:')
print('  0 = Malignant (cancerous)')
print('  1 = Benign (not cancerous)')
print('\nFirst 5 rows (showing a few columns):')
df[['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'target']].head()

In [ ]:
# Understand the structure
print('Total patients :', df.shape[0])
print('Total features :', df.shape[1] - 1)
print('\nAll 30 feature names:')
for i, name in enumerate(data.feature_names, 1):
    print(f'  {i:2d}. {name}')

print('\nMissing values:', df.isnull().sum().sum(), '(clean dataset!)')

## 📌 SECTION 3: Exploratory Data Analysis (EDA)

In [ ]:
# Check class balance
counts = df['target'].value_counts()
print('Class distribution:')
print(f'  Malignant (0): {counts[0]} patients ({counts[0]/len(df)*100:.1f}%)')
print(f'  Benign    (1): {counts[1]} patients ({counts[1]/len(df)*100:.1f}%)')

plt.figure(figsize=(7, 5))
colors = ['#D6336C', '#2E9E5B']
bars = plt.bar(['Malignant (0)', 'Benign (1)'], counts.values,
               color=colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x()+bar.get_width()/2, val+5, str(val),
             ha='center', fontweight='bold', fontsize=12)
plt.title('🎗️ Class Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Number of Patients')
plt.tight_layout()
plt.show()

In [ ]:
# Compare key features between Malignant and Benign
key_features = ['mean radius', 'mean texture', 'mean perimeter',
                'mean area', 'mean smoothness', 'mean concavity']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('🔬 Feature Distributions: Malignant vs Benign', fontsize=16, fontweight='bold')

for ax, feat in zip(axes.flat, key_features):
    malignant = df[df['target']==0][feat]
    benign    = df[df['target']==1][feat]
    ax.hist(malignant, bins=25, alpha=0.6, label='Malignant', color='#D6336C')
    ax.hist(benign,    bins=25, alpha=0.6, label='Benign',    color='#2E9E5B')
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.show()

print('📌 Notice: Malignant tumors tend to have LARGER radius, perimeter, area.')
print('   These features will be strong predictors!')

In [ ]:
# Correlation of key features with the target
corr_with_target = df.corr()['target'].drop('target').sort_values()

plt.figure(figsize=(9, 10))
colors_corr = ['#D6336C' if v < 0 else '#2E9E5B' for v in corr_with_target.values]
plt.barh(corr_with_target.index, corr_with_target.values, color=colors_corr, alpha=0.8)
plt.title('🔥 Feature Correlation with Target', fontsize=14, fontweight='bold')
plt.xlabel('Correlation (negative = predicts Malignant)')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('📌 Strong negative correlation = higher value points toward Malignant (0).')

## 📌 SECTION 4: Preprocessing

### Step 1 — Define X and y
### Step 2 — Split FIRST (stratified)
### Step 3 — Scale AFTER split (no data leakage!)

> Scaling is essential here — features range from 0.001 to 2500+, hugely different scales.

In [ ]:
# Step 1: Features and target
X = df.drop('target', axis=1)
y = df['target']

print('Feature range BEFORE scaling (notice the huge differences):')
print(f"  mean area:      {X['mean area'].min():.1f} to {X['mean area'].max():.1f}")
print(f"  mean smoothness: {X['mean smoothness'].min():.4f} to {X['mean smoothness'].max():.4f}")

# Step 2: Split FIRST (stratify to keep class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'\nTrain set: {X_train.shape[0]} patients')
print(f'Test  set: {X_test.shape[0]} patients')

# Step 3: Scale AFTER split
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)          # transform only on test

print('\n✅ Data split and scaled — no leakage!')
print(f'After scaling, mean area range: {X_train_scaled[:,3].min():.2f} to {X_train_scaled[:,3].max():.2f}')

## 📌 SECTION 5: Train 6 Classification Models

We train all 6 models and store their results for comparison.

In [ ]:
# Define all 6 models in a dictionary
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':         GaussianNB(),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42),
}

# Train each model and collect metrics
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred

    results.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).set_index('Model').round(4)
print('📊 All Models — Performance Comparison:')
print('=' * 65)
print(results_df)
print('=' * 65)
print(f"\n🏆 Highest Accuracy: {results_df['Accuracy'].idxmax()} ({results_df['Accuracy'].max():.4f})")
print(f"🎯 Highest Recall:   {results_df['Recall'].idxmax()} ({results_df['Recall'].max():.4f})")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(13, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(results_df))
width = 0.2
colors_m = ['#6C5CE7', '#0F9B8E', '#D6336C', '#E8845A']

for i, metric in enumerate(metrics):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=colors_m[i], alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(results_df.index, rotation=20, ha='right')
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Score')
ax.set_title('📊 Model Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 📌 SECTION 6: Confusion Matrix — The Medical View

> In cancer detection, a **False Negative** (saying 'safe' when it's cancer) is the most dangerous error.
> Let's look at the confusion matrix of the best model.

In [ ]:
# Pick the best model by recall (most important in medicine)
best_model_name = results_df['Recall'].idxmax()
y_pred_best = predictions[best_model_name]

cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', cbar=False,
            xticklabels=['Pred Malignant', 'Pred Benign'],
            yticklabels=['Actual Malignant', 'Actual Benign'],
            annot_kws={'size': 16, 'weight': 'bold'})
plt.title(f'Confusion Matrix — {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Explain each cell
tn, fp, fn, tp = cm.ravel()
print('📋 Reading the Confusion Matrix:')
print(f'  ✅ True Malignant caught : {tn}  (correctly found cancer)')
print(f'  ✅ True Benign          : {tp}  (correctly found safe)')
print(f'  ⚠️  False Positive       : {fp}  (said cancer, was safe - extra tests)')
print(f'  ❌ False Negative       : {fn}  (said safe, was cancer - DANGEROUS!)')
print(f'\n🎯 This model missed {fn} cancer case(s) out of {tn+fn} actual malignant tumors.')

## 📌 SECTION 7: ROC Curves — Compare All Models

> The **ROC curve** shows the tradeoff between catching true positives and false alarms.
> **AUC** (Area Under Curve) closer to 1.0 = better model.

In [ ]:
plt.figure(figsize=(9, 7))

for name, model in models.items():
    # Get probability of the positive class
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random guess')
plt.title('📈 ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 📌 SECTION 8: Cross-Validation — Reliable Comparison

> One split can be lucky. 5-Fold Cross-Validation gives a trustworthy estimate.

In [ ]:
# Scale the full dataset for CV
X_scaled_full = StandardScaler().fit_transform(X)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('5-Fold Cross-Validation Accuracy:')
print('=' * 50)
cv_means = {}
for name, model in models.items():
    scores = cross_val_score(model, X_scaled_full, y, cv=skf, scoring='accuracy')
    cv_means[name] = scores.mean()
    print(f'{name:<22}: {scores.mean():.4f} ± {scores.std():.4f}')

print('=' * 50)
best_cv = max(cv_means, key=cv_means.get)
print(f'\n🏆 Best by Cross-Validation: {best_cv} ({cv_means[best_cv]:.4f})')

## 📌 SECTION 9: Feature Importance (Random Forest)

> Which measurements matter most for detecting cancer?

In [ ]:
# Train a Random Forest and get feature importances
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
top10 = importances.tail(10)

plt.figure(figsize=(9, 6))
plt.barh(top10.index, top10.values, color='#6C5CE7', alpha=0.8)
plt.title('🌲 Top 10 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('📌 These features contribute most to distinguishing malignant from benign tumors.')

## 📌 SECTION 10: Make a Prediction on a New Patient

> Let's simulate using our trained model on a new tumor measurement.

In [ ]:
# Use the best model (Random Forest) for a demo prediction
final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X_train_scaled, y_train)

# Take 3 real test patients as examples
sample_indices = [0, 5, 10]
print('🔬 Predicting on 3 test patients:\n')

for idx in sample_indices:
    patient = X_test_scaled[idx].reshape(1, -1)
    pred = final_model.predict(patient)[0]
    proba = final_model.predict_proba(patient)[0]
    actual = y_test.iloc[idx]

    pred_label = 'Benign ✅' if pred == 1 else 'Malignant ⚠️'
    actual_label = 'Benign' if actual == 1 else 'Malignant'
    correct = '✓ CORRECT' if pred == actual else '✗ WRONG'

    print(f'Patient {idx}:')
    print(f'  Prediction : {pred_label}  (confidence: {max(proba)*100:.1f}%)')
    print(f'  Actual     : {actual_label}   →  {correct}')
    print()

---
## 📌 Summary & Key Takeaways

### What We Built
A machine learning system that detects breast cancer with **95%+ accuracy** using 6 different models.

### Model Comparison
| Model | Strength |
|-------|----------|
| **Logistic Regression** | Simple, fast, surprisingly strong baseline |
| **KNN** | Intuitive but sensitive to scaling |
| **Naive Bayes** | Fast, works well with independent features |
| **Decision Tree** | Interpretable rules, but can overfit |
| **Random Forest** | Usually best accuracy — many trees voting |
| **SVM** | Strong with clear margins between classes |

### The Most Important Lesson 🎗️
> In medical diagnosis, **Recall** (catching every real cancer) matters more than raw accuracy.
> A **False Negative** — telling a patient they're safe when they have cancer — can cost a life.
> Always look at the **confusion matrix**, not just the accuracy number.

### The Pipeline (memorize this!)
```
Load → EDA → Split → Scale → Train → Evaluate → Interpret
```

---

### 🎯 Assignment
1. Change KNN's `n_neighbors` from 5 to 3, 7, 11 — how does Recall change?
2. For the Decision Tree, try `max_depth` = 2, 5, 10, None — when does it overfit?
3. **Critical thinking:** If you were building a real hospital screening tool, which model would you deploy and why? Consider Recall over Accuracy.

---
*AIML Lab — Port City International University, Chattogram*